# Task 12: квантование, multi-fold training и выбор по P@20

Notebook читает готовый artifact. Долгий расчёт запускается из терминала через `scripts/run_task12_ranker_backtest.sh`; инструкция — `RANKER_BACKTEST.md`.
Полный run выполнен и проверен 2026-09-05: winner `fit_training`, 1,030 trees; canonical P@20 all/labeled 0.0050074443 / 0.0068163579, +877 hits (+4.5753%) к Task08. Review: `artifacts/task12_review_20260905_v1/findings.md`. Smoke не сравнивается с full score.

In [ ]:
from pathlib import Path
import polars as pl
from experiment_utils import read_json

run_id = 'task12_ranker_backtest_v1'
artifact = Path('artifacts') / run_id
if not (artifact / 'metrics.json').is_file():
    raise FileNotFoundError(f'{artifact}: сначала выполните terminal runner или укажите готовый smoke artifact')
config = read_json(artifact / 'config.json')
metrics = read_json(artifact / 'metrics.json')
print('mode:', config['mode'], '| canonical previously opened:', metrics['canonical_previously_opened'])
metrics['winner']

Число деревьев выбирается на rolling_3 по hits при неизменных denominators. Сравните политики при fixed tree count и их лучшие точки; Logloss служит диагностикой. Baseline — сохранённый Task08 на тех же ID-sampled пользователях.

In [ ]:
curves = pl.read_csv(artifact / 'selection_curves.csv')
curves.select('config_id', 'tree_count', 'final_hits', 'precision_at_20_all_targets',
              'precision_at_20_labeled_users', 'full_candidate_logloss').sort('config_id', 'tree_count')

In [ ]:
pl.DataFrame([{key: metrics[key] for key in (
    'precision_at_20_all_targets', 'precision_at_20_labeled_users', 'final_hits',
    'candidate_recall', 'coverage', 'candidate_oracle_p20_all_targets',
    'target_users', 'labeled_users', 'runtime_seconds', 'peak_memory_mb')}])

In [ ]:
metrics['comparisons']

In [ ]:
border_rows = []
for path in sorted((artifact / 'quantization').glob('*/quantization.json')):
    quantization = read_json(path)
    border_rows.extend({'pool': path.parent.name, **row} for row in quantization['features'])
pl.DataFrame(border_rows).select('pool', 'name', 'border_count').filter(
    pl.col('name').str.contains('72h') | (pl.col('border_count') == 0))